# v0 — Cascade baseline: Whisper-small → Gemma-2B-it

This is the **first iteration** of the project, run on a Colab L4 GPU. It is kept as the
historical record of the v0 baseline; its raw outputs are in
[`results/v0_cascade/raw_outputs.csv`](../results/v0_cascade/raw_outputs.csv).

> **Erratum (found during the v0 failure analysis — see [`docs/REPORT.md`](../docs/REPORT.md)).**
> The summary originally written for this notebook was wrong in three ways:
> 1. `evaluate`'s WER is a *fraction*: the printed `1.3936` is **139 % WER**, not 1.39 %.
>    It is dominated by Whisper hallucination loops on non-English clips.
> 2. The `other ⋈ reported` join duplicates clips that were reported more than once
>    (148 rows, **107 unique clips**).
> 3. ROUGE-L between a free-form generation and a one-word label is not a meaningful
>    score; the LLM stage is re-scored as classification in `scripts/score_v0.py`.
>    The prompt was also passed without Gemma's chat template, which is why the model
>    echoes the instruction scaffold.
>
> Corrected numbers: `python scripts/score_v0.py`. Improved pipeline: `scripts/run_cascade.py`.

Paths were Colab paths; they are parameterised as `CV_ROOT` so the notebook runs against
the local layout described in [`data/README.md`](../data/README.md).

# Implementation

### Import the libraries

In [16]:
%pip install -q transformers accelerate jiwer rouge_score evaluate bitsandbytes librosa

### Load the model with 4-bit quantization for reduced load

In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Chosen for efficiency (small size/high speed) and performance (instruction-tuned quality)
MODEL_NAME = "google/gemma-2b-it"

# Load the Tokenizer
llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load the Model with 4-bit quantization
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

print("LLM Model and Tokenizer Loaded.")
print(f"Model is running on device: {llm_model.device}")

LLM Model and Tokenizer Loaded.
Model is running on device: cuda:0


### Stage 1 of main pipeline

In [19]:
import torch
import librosa
from transformers import pipeline

whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small", # small version for speed
    device=llm_model.device # use gpu
)

def transcribe_audio_file(audio_path: str) -> str:
    """
    Transcribes an audio file using the Whisper ASR pipeline.
    """
    # The pipeline handles loading, feature extraction, and decoding
    result = whisper_pipe(audio_path)
    return result["text"].strip()

Device set to use cuda:0


### Stage 2 of main pipeline

In [20]:
def reason_on_transcript(transcript: str, task_prompt: str, llm_model, llm_tokenizer) -> str:
    """
    Feeds the transcript and task to the LLM for reasoning.
    """
    full_prompt = (
        f"You are an expert reasoning assistant. Based ONLY on the following audio transcript, "
        f"perform the requested task.\n\n"
        f"--- TRANSCRIPT ---\n"
        f"{transcript}\n"
        f"------------------\n\n"
        f"TASK: {task_prompt}"
    )

    # Tokenize the input prompt
    inputs = llm_tokenizer(
        full_prompt,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(llm_model.device)

    # Generate the response
    with torch.no_grad():
        output_tokens = llm_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            pad_token_id=llm_tokenizer.eos_token_id
        )

    response = llm_tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    # Remove the input prompt from the response (common LLM generation issue)
    if response.startswith(full_prompt):
        response = response[len(full_prompt):].strip()

    return response

### Dataset location

In [21]:
# Extract the Common Voice delta once (see data/README.md):
#   tar -xzf cv-corpus-22.0-delta-2025-06-20-en.tar.gz -C data/raw
CV_ROOT = "../data/raw/cv-corpus-22.0-delta-2025-06-20"

### Clean and prepare the dataframe with transcribed sentences out of audio files

In [22]:
import pandas as pd

clip_dur = pd.read_csv(f'{CV_ROOT}/en/clip_durations.tsv', sep='\t')
clip_dur.head()

,clip,duration[ms]
0,common_voice_en_43160279.mp3,5868
1,common_voice_en_43200228.mp3,8676
2,common_voice_en_43238638.mp3,8208
3,common_voice_en_43198702.mp3,8496
4,common_voice_en_43197149.mp3,6480


In [23]:
import pandas as pd

other = pd.read_csv(f'{CV_ROOT}/en/other.tsv', sep='\t')
other.head()

,client_id,path,sentence_id,sentence,sentence_domain,up_votes,down_votes,age,gender,accents,variant,locale,segment
0,e192d7cd52e62566150a257e0d993b5b0a03ba298311c8...,common_voice_en_42696072.mp3,f889b83dc83795676e7d33fd96e09079512211dd61f9cb...,But then something even more extraordinary hap...,NaN,0,0,NaN,NaN,NaN,NaN,en,NaN
1,5e408ba4e2f9562f17efc82c2c7d3d91f2b53729e4506c...,common_voice_en_42696165.mp3,f88dd711b7b99232fdb1fe93556c2c21a9ff0c36bdf1da...,The division was then commanded by Major Gener...,NaN,0,0,NaN,NaN,NaN,NaN,en,NaN
2,5e408ba4e2f9562f17efc82c2c7d3d91f2b53729e4506c...,common_voice_en_42696166.mp3,f8990814817b6b84c2d9f7f44992b6ce1455ae8c751a7b...,"Her husband was Li Shi, the last emperor of Ch...",NaN,0,0,NaN,NaN,NaN,NaN,en,NaN
3,5e408ba4e2f9562f17efc82c2c7d3d91f2b53729e4506c...,common_voice_en_42696167.mp3,f88e6e1e48bb514d9da92b461eeb6826b9aed1220a973a...,"London advances against Ludington, Michigan, c...",NaN,0,0,NaN,NaN,NaN,NaN,en,NaN
4,5e408ba4e2f9562f17efc82c2c7d3d91f2b53729e4506c...,common_voice_en_42696168.mp3,f88fa2e6d0ab1f9f71d798176d3b5ae5439180931f1371...,But his books were never best-sellers.,NaN,0,0,NaN,NaN,NaN,NaN,en,NaN


In [24]:
import pandas as pd

reported = pd.read_csv(f'{CV_ROOT}/en/reported.tsv', sep='\t')
reported.head()

,sentence_id,sentence,locale,reason
0,f88c245223cf2a796c4c98f882aff9f7269cbd54d3a1cd...,The Welsh Office was created to execute govern...,en,offensive-language
1,f89e9101e0a488771f1d1612a238924f9a09a00f6b6e2d...,Before deciding to become an architect he had ...,en,grammar-or-spelling
2,f895a6eaec03f6cfbbf717f6a3610ac9d7ee9f1d645190...,A faint ghost crater lies in the mare to the n...,en,grammar-or-spelling
3,f87bbb0bdb343b48aa135afee217f2c445c735be7438b9...,The legs are short and black as well.,en,grammar-or-spelling
4,f8a0fb027a553201a62b5fcb750e66e6fdd3f588bb5c10...,It helps focus on the subject and not its envi...,en,grammar-or-spelling


In [25]:
reason_others = pd.merge(other, reported, on='sentence_id', how="inner")
reason_others = reason_others[['sentence_id', 'reason', 'sentence_x', 'path', 'gender', 'accents']]

reason_others.head()

,sentence_id,reason,sentence_x,path,gender,accents
0,f88c245223cf2a796c4c98f882aff9f7269cbd54d3a1cd...,offensive-language,The Welsh Office was created to execute govern...,common_voice_en_42706193.mp3,male_masculine,United States English
1,f87bbb0bdb343b48aa135afee217f2c445c735be7438b9...,grammar-or-spelling,The legs are short and black as well.,common_voice_en_42710400.mp3,NaN,NaN
2,f8a0fb027a553201a62b5fcb750e66e6fdd3f588bb5c10...,grammar-or-spelling,It helps focus on the subject and not its envi...,common_voice_en_42710509.mp3,male_masculine,Filipino
3,f895a6eaec03f6cfbbf717f6a3610ac9d7ee9f1d645190...,grammar-or-spelling,A faint ghost crater lies in the mare to the n...,common_voice_en_42713213.mp3,male_masculine,Filipino
4,f89e9101e0a488771f1d1612a238924f9a09a00f6b6e2d...,grammar-or-spelling,Before deciding to become an architect he had ...,common_voice_en_42713267.mp3,male_masculine,Filipino


In [26]:
clip_dur = clip_dur.rename(columns={'clip': 'path'})

reason_others = pd.merge(reason_others, clip_dur, on='path', how="inner")
unique_columns = ~reason_others.columns.duplicated()

# Select only the columns with unique names
reason_others = reason_others.loc[:, unique_columns]
reason_others.head()

,sentence_id,reason,sentence_x,path,gender,accents,duration[ms]
0,f88c245223cf2a796c4c98f882aff9f7269cbd54d3a1cd...,offensive-language,The Welsh Office was created to execute govern...,common_voice_en_42706193.mp3,male_masculine,United States English,7200
1,f87bbb0bdb343b48aa135afee217f2c445c735be7438b9...,grammar-or-spelling,The legs are short and black as well.,common_voice_en_42710400.mp3,NaN,NaN,5148
2,f8a0fb027a553201a62b5fcb750e66e6fdd3f588bb5c10...,grammar-or-spelling,It helps focus on the subject and not its envi...,common_voice_en_42710509.mp3,male_masculine,Filipino,5688
3,f895a6eaec03f6cfbbf717f6a3610ac9d7ee9f1d645190...,grammar-or-spelling,A faint ghost crater lies in the mare to the n...,common_voice_en_42713213.mp3,male_masculine,Filipino,5472
4,f89e9101e0a488771f1d1612a238924f9a09a00f6b6e2d...,grammar-or-spelling,Before deciding to become an architect he had ...,common_voice_en_42713267.mp3,male_masculine,Filipino,6300


In [27]:
reason_others.to_csv('../data/processed/reason_others_v0.csv')

In [28]:
df = reason_others.copy()
df['reason'].value_counts()

,count
reason,
different-language,45
grammar-or-spelling,36
sc-different-language,25
offensive-language,17
difficult-pronounce,9
difficult-pronunciation,9
Tiktok,1
I have failed to record the sentences I have read out loud.,1
tiktok,1


In [38]:
import pandas as pd
import evaluate
from tqdm import tqdm
import re
import os
from typing import Dict, Any, Tuple

# Load WER and ROUGE-L from the 'evaluate' library
# NOTE: Ensure 'pip install evaluate jiwer rouge_score' has been run
wer_metric = evaluate.load("wer")
rouge_metric = evaluate.load("rouge")

def run_evaluation_pipeline_with_dataframe(
    df: pd.DataFrame,
    task_prompt: str,
    llm_model,
    llm_tokenizer,
    audio_clip_base_path: str
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    """
    Runs the ASR -> LLM pipeline across a Pandas DataFrame, calculates metrics,
    and returns a detailed results DataFrame for reporting.

    Args:
        df: Pandas DataFrame containing the evaluation data.
        task_prompt: The static reasoning prompt given to the LLM.
        llm_model: The loaded LLM model.
        llm_tokenizer: The loaded LLM tokenizer.
        audio_clip_base_path: The directory path where the audio files ('path' column) reside.

    Returns:
        A tuple containing:
        1. A dictionary of overall evaluation metrics.
        2. A Pandas DataFrame with ground truth, ASR transcript, and LLM output.
    """

    asr_predictions = []
    asr_references = []
    llm_predictions = []
    llm_references = []

    # NEW: List to store detailed results for the final DataFrame
    detailed_results = []

    for index, sample in tqdm(df.iterrows(), total=len(df), desc="Evaluating Pipeline"):
        # Create a dictionary to hold all results for this row
        row_data = sample.to_dict()
        row_data['asr_transcript'] = None
        row_data['llm_reasoning_output'] = None

        audio_filename = sample['path']
        audio_path = os.path.join(audio_clip_base_path, audio_filename)

        # Flag to track successful processing
        processed_successfully = False

        try:
            if not os.path.exists(audio_path):
                # print(f"Audio file not found at: {audio_path}. Skipping.")
                continue

            # 1. ASR Step (Whisper)
            transcript = transcribe_audio_file(audio_path)
            row_data['asr_transcript'] = transcript

            # 2. LLM Reasoning Step (Gemma)
            llm_output = reason_on_transcript(
                transcript=transcript,
                task_prompt=task_prompt,
                llm_model=llm_model,
                llm_tokenizer=llm_tokenizer
            )
            row_data['llm_reasoning_output'] = llm_output

            # Append data for metrics calculation
            asr_predictions.append(transcript)
            asr_references.append(sample['sentence_x'])
            llm_predictions.append(llm_output)
            llm_references.append(sample['reason'])

            processed_successfully = True

        except Exception as e:
            print(f"Error processing sample {index} (File: {audio_filename}): {e}. Skipping.")
            # Even if processing failed, we still want to record the sample data in the final DF
            row_data['error'] = str(e)

        # Append the row data (whether successful or not) to the detailed results list
        # Only append rows that were part of the initial DataFrame
        detailed_results.append(row_data)

    # --- Create Detailed Results DataFrame ---
    result_df = pd.DataFrame(detailed_results)


    # --- 3. Metric Calculation ---
    if not asr_predictions:
        metrics = {"total_samples_evaluated": 0, "asr_wer": None, "reasoning_rouge_l_fscore": None}
        return metrics, result_df

    # A. Calculate Word Error Rate (WER)
    wer_score = wer_metric.compute(predictions=asr_predictions, references=asr_references)

    # B. Calculate Reasoning Score (ROUGE-L)
    rouge_results = rouge_metric.compute(
        predictions=llm_predictions,
        references=llm_references,
        rouge_types=["rougeL"]
    )

    # Safe ROUGE Access (to handle varying output structures)
    try:
        if "rougeL" in rouge_results and isinstance(rouge_results["rougeL"], dict):
            rouge_l_fscore = rouge_results["rougeL"]["fmeasure"]
        elif 'rougeL' in rouge_results and isinstance(rouge_results['rougeL'], float):
             rouge_l_fscore = rouge_results['rougeL']
        else:
             raise ValueError("Unexpected ROUGE result format.")
    except (KeyError, ValueError) as e:
         print(f"ROUGE calculation error: {e}. Setting score to None.")
         rouge_l_fscore = None


    # --- 4. Return Metrics and Detailed DataFrame ---
    metrics = {
        "total_samples_evaluated": len(asr_predictions),
        "asr_wer": round(wer_score, 4),
        "reasoning_rouge_l_fscore": round(rouge_l_fscore, 4) if rouge_l_fscore is not None else None,
    }

    return metrics, result_df

# Assuming reason_others, llm_model, llm_tokenizer are defined globally
# BASE_AUDIO_PATH is defined globally

BASE_AUDIO_PATH = f"{CV_ROOT}/en/clips"

# Instruct the llm
REASONING_TASK = """
You are a reasoning model that analyzes ASR-transcribed text, which may include errors such as misspellings,
foreign words, or incomplete phrases. Your goal is to:
1. Understand the intended meaning of the text, even if grammar or spelling mistakes exist. If it's foreign language other than English, identify it as 'different-language'.
2. Identify the main topic or issue discussed.
3. Extract the three most important facts, reasons, or categories mentioned.
4. If the transcription seems unclear (e.g., due to multiple languages or offensive words), make your best
contextual inference without adding unrelated information.

Return your output as:
Topic: <short topic phrase>
Reason: <identify the tone of the sound>
Fact 1: <first fact or category>
Fact 2: <second fact or category>
Fact 3: <third fact or category>
"""


#  Evaluation
print("\n" + "="*50)
print("Starting Pipeline Evaluation ")
print("="*50)



evaluation_results, result_df = run_evaluation_pipeline_with_dataframe(
    df=reason_others,
    task_prompt=REASONING_TASK,
    llm_model=llm_model,
    llm_tokenizer=llm_tokenizer,
    audio_clip_base_path=BASE_AUDIO_PATH
)


print("\n--- Final Evaluation Scores ---")
print(f"Total Samples: {evaluation_results['total_samples_evaluated']}")
print(f"ASR Word Error Rate (WER): {evaluation_results['asr_wer']}")
print(f"LLM Reasoning ROUGE-L F1 Score: {evaluation_results['reasoning_rouge_l_fscore']}")
print("-----------------------------\n")

print("\n--- Detailed Results DataFrame (First 5 Rows) ---")
print(result_df[['sentence_x', 'reason', 'asr_transcript', 'llm_reasoning_output']].head().to_markdown())
print("-----------------------------\n")





Starting Pipeline Evaluation 



--- Final Evaluation Scores ---
Total Samples: 148
ASR Word Error Rate (WER): 1.3936
LLM Reasoning ROUGE-L F1 Score: 0.0287
-----------------------------


--- Detailed Results DataFrame (First 5 Rows) ---
|    | sentence_x                                                                      | reason              | asr_transcript                                                                   | llm_reasoning_output                                                                                                                                         |
|---:|:--------------------------------------------------------------------------------|:--------------------|:---------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------|
|  0 | The Welsh Office was created to execute government policy in Wales.             

In [39]:
result_df.to_csv('../results/v0_cascade/raw_outputs.csv')

# Interface using Gradio

In [34]:
import gradio as gr
import os

def process_audio_and_reason(audio_path: str) -> tuple[str, str]:
    """
    Takes the audio file path (provided by Gradio), runs ASR, and then runs the LLM.

    Args:
        audio_path: Path to the temporary WAV/MP3 file recorded by Gradio.

    Returns:
        A tuple containing the ASR transcript and the LLM's reasoning.
    """
    if not audio_path:
        return "Please record or upload an audio file.", "No analysis performed."

    print(f"Processing audio from: {audio_path}")

    try:
        # 1. ASR Step (Whisper)
        transcript = transcribe_audio_file(audio_path)

        # 2. LLM Reasoning Step
        llm_output = reason_on_transcript(
            transcript=transcript,
            task_prompt=REASONING_TASK,
            llm_model=llm_model,
            llm_tokenizer=llm_tokenizer
        )

        return transcript, llm_output

    except Exception as e:
        error_msg = f"An error occurred during processing: {e}"
        print(error_msg)
        return "Transcription failed.", error_msg

In [35]:

iface = gr.Interface(
    fn=process_audio_and_reason,

    # --- Input Component ---
    inputs=gr.Audio(sources=["microphone", "upload"], type="filepath", label="Input Audio (Speak or Upload)"),

    # --- Output Components ---
    outputs=[
        gr.Textbox(label="1. ASR Transcript (Whisper Output)", lines=3),
        gr.Textbox(label="2. LLM Reasoning/Response", lines=7)
    ],

    # --- Interface Settings ---
    title="🎙️ ASR + LLM Reasoning Pipeline Demo",
    description="Speak into the microphone or upload an audio clip. The audio will be transcribed by Whisper, and the transcript will be analyzed by the LLM (e.g., Gemma/Mistral) to provide reasoning.",
    allow_flagging="never"
)

# Launch
print("\nLaunching Gradio interface locally...")
iface.launch(inbrowser=True, share=False)


Launching Gradio interface locally...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.
